# 01 - Full Dataset Exploration

**Project:** E-Commerce User Behavior Analysis & Recommendation System  
**Notebook purpose:** Full exploration of both raw data files across all 110 million rows. Provides accurate data quality statistics that inform the preprocessing pipeline. 
For a quick sample-based exploration of the schema, see `00_data_exploration.ipynb`.  

---

## Environment Setup

This notebook was run on **Kaggle Notebooks** using the dataset attached directly 
from the Kaggle dataset page. No local setup or downloads are required to run it.

### To reproduce this notebook

1. Go to the repository on GitHub:  
   [ecommerce-behavior-analysis](https://github.com/halleepham/ecommerce-behavior-analysis)
2. Download `notebooks/01_full_dataset_exploration.ipynb`
3. Go to the dataset page on Kaggle:  
   [E-Commerce Behavior Data from Multi-Category Store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store)
4. Click the three dots, then **New Notebook** to open a fresh Kaggle notebook with the dataset already attached
5. Click 'File' -> 'Import Notebook' and select the downloaded file. The dataset will already be attached
6. Run all cells top to bottom

### Data path

All cells in this notebook use the following path to access the raw data:

    /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store

### Python version and key libraries

| Library | Version |
|---|---|
| Python | 3.12.12 |
| pandas | 2.3.3 |
| numpy | 2.0.2 |

---

## Scope

This notebook reads both complete raw files (`2019-Oct.csv` and `2019-Nov.csv`) 
in chunks of 100,000 rows at a time to avoid memory issues. Each section collects 
statistics across the full dataset and compares October and November side by side.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from tqdm import tqdm

# Data paths
data_path = '/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store'
oct_file  = f'{data_path}/2019-Oct.csv'
nov_file  = f'{data_path}/2019-Nov.csv'

# Constants
CHUNK_SIZE = 100_000

# Verify files exist 
print(f'October  file exists : {os.path.exists(oct_file)}')
print(f'November file exists : {os.path.exists(nov_file)}\n')

print(f'Python  : {sys.version}')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')

October  file exists : True
November file exists : True

Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
pandas  : 2.3.3
numpy   : 2.0.2


In [2]:
def collect_shape_stats(chunk, stats):
    """Collect row count, unique sessions, and null session ID count."""
    stats['total_rows'] += len(chunk)
    stats['sessions'].update(chunk['user_session'].dropna().unique())
    stats['null_sessions'] += chunk['user_session'].isna().sum()

In [3]:
def collect_missing_stats(chunk, stats):
    """Collect null count per column."""
    for col in chunk.columns:
        stats['missing'][col] = stats['missing'].get(col, 0) + chunk[col].isna().sum()

In [4]:
def collect_category_recovery_stats(chunk, stats):
    """Collect product and category IDs where category_code is missing or present."""
    missing = chunk[chunk['category_code'].isna()]
    present = chunk[chunk['category_code'].notna()]
    stats['products_missing_code'].update(missing['product_id'].unique())
    stats['products_with_code'].update(present['product_id'].unique())
    stats['ids_missing_code'].update(missing['category_id'].unique())
    stats['ids_with_code'].update(present['category_id'].unique())

In [5]:
def collect_event_stats(chunk, stats):
    """Collect event type distribution."""
    counts = chunk['event_type'].value_counts()
    for event, count in counts.items():
        stats['event_counts'][event] = stats['event_counts'].get(event, 0) + count

In [6]:
def collect_duplicate_stats(chunk, stats):
    """Collect duplicate row count."""
    stats['duplicates'] += chunk.duplicated().sum()

In [7]:
def collect_unique_stats(chunk, stats):
    """Collect unique users, products, brands, and top level categories."""
    stats['users'].update(chunk['user_id'].dropna().unique())
    stats['products'].update(chunk['product_id'].dropna().unique())
    stats['brands'].update(chunk['brand'].dropna().unique())
    
    # Extract top level category from category_code e.g. 'electronics' from 'electronics.smartphone'
    top_cats = chunk['category_code'].dropna().str.split('.').str[0]
    stats['categories'].update(top_cats.unique())

In [8]:
def collect_time_stats(chunk, stats):
    """Collect earliest and latest event_time and unique days."""
    # Parse timestamps in this chunk
    times = pd.to_datetime(chunk['event_time'], utc=True, errors='coerce').dropna()
    if len(times) == 0:
        return
    
    chunk_min = times.min()
    chunk_max = times.max()
    
    # Update running min and max
    if stats['min_time'] is None or chunk_min < stats['min_time']:
        stats['min_time'] = chunk_min
    if stats['max_time'] is None or chunk_max > stats['max_time']:
        stats['max_time'] = chunk_max
    
    stats['unique_days'].update(times.dt.date.unique())

In [9]:
def collect_purchase_stats(chunk, stats):
    """Collect users and products that have at least one purchase."""
    purchases = chunk[chunk['event_type'] == 'purchase']
    stats['users_with_purchase'].update(purchases['user_id'].dropna().unique())
    stats['products_with_purchase'].update(purchases['product_id'].dropna().unique())

In [10]:
def collect_price_stats(chunk, stats):
    """Collect price distribution statistics and zero price counts."""
    prices = chunk['price'].dropna()
    
    stats['price_sum']   += prices.sum()
    stats['price_count'] += len(prices)
    stats['zero_prices'] += (prices == 0).sum()
    stats['price_min']    = min(stats['price_min'], prices.min()) if len(prices) > 0 else stats['price_min']
    stats['price_max']    = max(stats['price_max'], prices.max()) if len(prices) > 0 else stats['price_max']
    
    # Track zero price products
    zero_price_products = chunk[chunk['price'] == 0]['product_id'].dropna()
    stats['zero_price_products'].update(zero_price_products.unique())
    
    # Store prices for median and std calculation — sample to avoid memory issues
    if len(stats['price_sample']) < 500_000:
        stats['price_sample'].extend(prices.sample(
            min(len(prices), 1000), random_state=42).tolist())

In [11]:
def analyze_file(filepath):
    """
    Read a CSV file in chunks and collect all statistics in a single pass.
    Calls each smaller collect function on every chunk.
    Returns a dictionary of all collected statistics.
    """
    # Initialize stats dictionary with default values
    stats = {
        # Shape
        'total_rows'           : 0,
        'sessions'             : set(),
        'null_sessions'        : 0,
        # Missing values
        'missing'              : {},
        # Category_code recovery
        'products_missing_code' : set(),
        'products_with_code'    : set(),
        'ids_missing_code'      : set(),
        'ids_with_code'         : set(),
        # Event types
        'event_counts'         : {},
        # Duplicates
        'duplicates'           : 0,
        # Unique counts
        'users'                : set(),
        'products'             : set(),
        'brands'               : set(),
        'categories'           : set(),
        # Time
        'min_time'             : None,
        'max_time'             : None,
        'unique_days'          : set(),
        # Purchase behavior
        'users_with_purchase'  : set(),
        'products_with_purchase': set(),
        # Pricing
        'price_sum'            : 0,
        'price_count'          : 0,
        'price_min'            : float('inf'),
        'price_max'            : float('-inf'),
        'zero_prices'          : 0,
        'zero_price_products'  : set(),
        'price_sample'         : [],
    }

    for chunk in tqdm(pd.read_csv(filepath, chunksize=CHUNK_SIZE),
                      desc=os.path.basename(filepath)):
        collect_shape_stats(chunk, stats)
        collect_missing_stats(chunk, stats)
        collect_category_recovery_stats(chunk, stats)
        collect_event_stats(chunk, stats)
        collect_duplicate_stats(chunk, stats)
        collect_unique_stats(chunk, stats)
        collect_time_stats(chunk, stats)
        collect_purchase_stats(chunk, stats)
        collect_price_stats(chunk, stats)

    return stats

In [12]:
# Analyze both files
print('Analyzing October...')
oct_stats = analyze_file(oct_file)
print('\nAnalyzing November...')
nov_stats = analyze_file(nov_file)
print('\nDone. Run the cells below to view results.')

Analyzing October...


2019-Oct.csv: 425it [09:29,  1.34s/it]



Analyzing November...


2019-Nov.csv: 676it [31:26,  2.79s/it]


Done. Run the cells below to view results.


# Results

## Dataset Shape

Count total rows, unique sessions, and average rows per session across the full dataset.

In [13]:
print('--- Dataset Shape ---\n')
print(f'{"Metric":<25} {"October":>15} {"November":>15}')
print(f'{"-" * 55}')
print(f'{"Total rows":<25} {oct_stats["total_rows"]:>15,} {nov_stats["total_rows"]:>15,}')
print(f'{"Total sessions":<25} {len(oct_stats["sessions"]):>15,} {len(nov_stats["sessions"]):>15,}')
print(f'{"Null session IDs":<25} {oct_stats["null_sessions"]:>15,} {nov_stats["null_sessions"]:>15,}')
print(f'{"Avg rows per session":<25} {oct_stats["total_rows"]/len(oct_stats["sessions"]):>15.2f} {nov_stats["total_rows"]/len(nov_stats["sessions"]):>15.2f}')

# Calculate totals for proportion calculations
total_rows     = oct_stats['total_rows'] + nov_stats['total_rows']
total_sessions = len(oct_stats['sessions']) + len(nov_stats['sessions'])

print(f'\n--- Proportions ---\n')
print(f'{"Metric":<35} {"October":>10} {"November":>10}')
print(f'{"-" * 55}')
print(f'{"% of total rows":<35} {oct_stats["total_rows"]/total_rows*100:>9.1f}% {nov_stats["total_rows"]/total_rows*100:>9.1f}%')
print(f'{"% of total sessions":<35} {len(oct_stats["sessions"])/total_sessions*100:>9.1f}% {len(nov_stats["sessions"])/total_sessions*100:>9.1f}%')

--- Dataset Shape ---

Metric                            October        November
-------------------------------------------------------
Total rows                     42,448,764      67,501,979
Total sessions                  9,244,421      13,776,050
Null session IDs                        2              10
Avg rows per session                 4.59            4.90

--- Proportions ---

Metric                                 October   November
-------------------------------------------------------
% of total rows                          38.6%      61.4%
% of total sessions                      40.2%      59.8%


### Shape Observations

| Metric | October | November |
|---|---|---|
| Total rows | 42,448,764 | 67,501,979 |
| Total sessions | 9,244,421 | 13,776,050 |
| Null session IDs | 2 | 10 |
| Avg rows per session | 4.59 | 4.90 |

* November has more rows and sessions than October (61.4% of all rows and 59.8% of all sessions come from November)
* The average session length is similar between months (4.59 vs 4.90 rows), suggesting consistent user behavior across both months
* The number of null session IDs are small. 2 rows in October and 10 rows in November out of 110 million total rows. These rows will be excluded from session-based sampling as they cannot belong to any session.

## Missing Values

Missing value counts across the full dataset for both files.

In [14]:
print('--- Missing Values ---\n')
print(f'{"Column":<20} {"October":>15} {"October %":>10} {"November":>15} {"November %":>10}')
print(f'{"-" * 70}')

# Get all columns from both files
all_cols = sorted(set(list(oct_stats['missing'].keys()) + list(nov_stats['missing'].keys())))

for col in all_cols:
    oct_missing = oct_stats['missing'].get(col, 0)
    nov_missing = nov_stats['missing'].get(col, 0)
    oct_pct = (oct_missing / oct_stats['total_rows']) * 100
    nov_pct = (nov_missing / nov_stats['total_rows']) * 100
    print(f'{col:<20} {oct_missing:>15,} {oct_pct:>9.1f}% {nov_missing:>15,} {nov_pct:>9.1f}%')

--- Missing Values ---

Column                       October  October %        November November %
----------------------------------------------------------------------
brand                      6,117,080      14.4%       9,224,078      13.7%
category_code             13,515,609      31.8%      21,898,171      32.4%
category_id                        0       0.0%               0       0.0%
event_time                         0       0.0%               0       0.0%
event_type                         0       0.0%               0       0.0%
price                              0       0.0%               0       0.0%
product_id                         0       0.0%               0       0.0%
user_id                            0       0.0%               0       0.0%
user_session                       2       0.0%              10       0.0%


### Missing Values Observations

| Column | October | October % | November | November % |
|---|---|---|---|---|
| `brand` | 6,117,080 | 14.4% | 9,224,078 | 13.7% |
| `category_code` | 13,515,609 | 31.8% | 21,898,171 | 32.4% |
| `user_session` | 2 | 0.0% | 10 | 0.0% |
| All other columns | 0 | 0.0% | 0 | 0.0% |

* Three columns have missing values: `category_code`, `brand`, and `user_session`
* `category_code` is missing in roughly 1 in 3 rows in both months. This is consistent. This may show that this is not a required or focused field in the source system. Check later if `category_code` has any relation to `category_id`
* `brand` is missing in roughly 1 in 7 rows. This is also consistent across both months
* `user_session` has a small number of missing values: 2 rows in October and 10 rows in November out of 110 million total rows
* The consistency between October and November across all three columns suggests these are systematic patterns in the data, not data collection errors
* Our earlier estimate from the 100,000 row sample (~32% and ~14%) was accurate, confirming the sample was representative for **this metric**
* `category_code` and `brand` will not be dropped, rows with missing values are still valid events. Missing values will be filled with 'unknown' during preprocessing. We'll see if `category_code` has any relation to `category_id`
* `user_session` null rows will be excluded from session-based sampling as they cannot belong to any session by definition

### Investigating category_code Recovery
Since `category_code` has ~32% missing values but `category_id` has none, we 
want to check if the missing `category_code` values can be recovered from other 
columns. We investigate two potential recovery paths:

1. **Via product_id** -- if the same product appears in some rows with a `category_code` and in other rows without, we can fill in the missing values by looking up the product's known category code
2. **Via category_id** -- if a `category_id` always maps to the same `category_code`, we can use that mapping to fill in missing values

Both files are checked together since a product may have a `category_code` in 
one month but not the other.

In [24]:
# Combine both months for a complete picture
all_products_missing = oct_stats['products_missing_code'] | nov_stats['products_missing_code']
all_products_with    = oct_stats['products_with_code']    | nov_stats['products_with_code']
all_ids_missing      = oct_stats['ids_missing_code']      | nov_stats['ids_missing_code']
all_ids_with         = oct_stats['ids_with_code']         | nov_stats['ids_with_code']

# Check overlap
recoverable_products = all_products_missing & all_products_with
recoverable_ids      = all_ids_missing      & all_ids_with

print('--- category_code Recovery via product_id ---\n')
print(f'{"Unique products with missing category_code":<45} {len(all_products_missing):>10,}')
print(f'{"Unique products with present category_code":<45} {len(all_products_with):>10,}')
print(f'{"Products appearing both with and without code":<45} {len(recoverable_products):>10,}')

print(f'\n--- category_code Recovery via category_id ---\n')
print(f'{"Unique category_ids with missing category_code":<45} {len(all_ids_missing):>10,}')
print(f'{"Unique category_ids with present category_code":<45} {len(all_ids_with):>10,}')
print(f'{"Category IDs appearing both with and without":<45} {len(recoverable_ids):>10,}')

--- category_code Recovery via product_id ---

Unique products with missing category_code       118,185
Unique products with present category_code        88,691
Products appearing both with and without code          0

--- category_code Recovery via category_id ---

Unique category_ids with missing category_code        414
Unique category_ids with present category_code        277
Category IDs appearing both with and without           0


### category_code Recovery Observations

| Metric | Count |
|---|---|
| Unique products with missing category_code | 118,185 |
| Unique products with present category_code | 88,691 |
| Products appearing both with and without code | 0 |
| Unique category_ids with missing category_code | 414 |
| Unique category_ids with present category_code | 277 |
| Category IDs appearing both with and without | 0 |

* Neither recovery path worked. 0 products are recoverable via product_id and 0 are recoverable via category_id
* There are actually more products without a category code (118,185) than with one (88,691), meaning the majority of distinct products in this store have no category code at all
* This suggests there are two completely separate groups of products in the store: those that have been assigned a category code and those that have not
* This confirms that missing `category_code` values cannot be recovered from anywhere else in the dataset
* Missing `category_code` values will be filled with 'unknown' during preprocessing since there is no way to recover them from other columns
* `category_id` may be kept as a column since it may still be useful as a numeric identifier (not human-readable) even without a corresponding `category_code`

## Event Type Distribution

Event type counts and percentages across the full dataset for both files.

In [16]:
print('--- Event Type Distribution ---\n')
print(f'{"Event Type":<15} {"October":>15} {"October %":>10} {"November":>15} {"November %":>10}')
print(f'{"-" * 65}')

# Get all event types from both files
all_events = sorted(set(list(oct_stats['event_counts'].keys()) + 
                        list(nov_stats['event_counts'].keys())))

for event in all_events:
    oct_count = oct_stats['event_counts'].get(event, 0)
    nov_count = nov_stats['event_counts'].get(event, 0)
    oct_pct = (oct_count / oct_stats['total_rows']) * 100
    nov_pct = (nov_count / nov_stats['total_rows']) * 100
    print(f'{event:<15} {oct_count:>15,} {oct_pct:>9.1f}% {nov_count:>15,} {nov_pct:>9.1f}%')

# Cart to purchase conversion rates
oct_conversion = (oct_stats['event_counts'].get('purchase', 0) /
                  oct_stats['event_counts'].get('cart', 0)) * 100
nov_conversion = (nov_stats['event_counts'].get('purchase', 0) /
                  nov_stats['event_counts'].get('cart', 0)) * 100

print(f'\n--- Cart to Purchase Conversion ---\n')
print(f'{"Metric":<35} {"October":>15} {"November":>15}')
print(f'{"-" * 65}')
print(f'{"Cart events":<35} {oct_stats["event_counts"].get("cart", 0):>15,} {nov_stats["event_counts"].get("cart", 0):>15,}')
print(f'{"Purchase events":<35} {oct_stats["event_counts"].get("purchase", 0):>15,} {nov_stats["event_counts"].get("purchase", 0):>15,}')
print(f'{"Cart to purchase conversion":<35} {oct_conversion:>14.1f}% {nov_conversion:>14.1f}%')

--- Event Type Distribution ---

Event Type              October  October %        November November %
-----------------------------------------------------------------
cart                    926,516       2.2%       3,028,930       4.5%
purchase                742,849       1.7%         916,939       1.4%
view                 40,779,399      96.1%      63,556,110      94.2%

--- Cart to Purchase Conversion ---

Metric                                      October        November
-----------------------------------------------------------------
Cart events                                 926,516       3,028,930
Purchase events                             742,849         916,939
Cart to purchase conversion                   80.2%           30.3%


### Event Type Distribution Observations

| Event Type | October | October % | November | November % |
|---|---|---|---|---|
| `view` | 40,779,399 | 96.1% | 63,556,110 | 94.2% |
| `cart` | 926,516 | 2.2% | 3,028,930 | 4.5% |
| `purchase` | 742,849 | 1.7% | 916,939 | 1.4% |

* Views are the majority in both months as expected. Most users browse without buying
* November has a noticeably higher cart rate (4.5% vs 2.2%), which may be from Black Friday and holiday shopping (users add items to cart to track prices or save for later). This will be looked into later.
* Purchase rate is slightly lower in November (1.4% vs 1.7%) despite more cart additions. This could indicate more browsing and comparison shopping behavior during the holiday season
* The overall cart-to-purchase conversion rate differs significantly between months (October 80.2% and November 30.3%)
* This dramatic difference in cart-to-purchase conversion between months is an important finding. It might suggest that November users add items to cart and wait instead of adding to the cart to immediately buy, which is consistent with holiday shopping behavior. But this will have to be looked into further.
* These differences confirm that including both months in the dataset is important for capturing the full range of user behavior patterns and how they change through seasons.

## Duplicate Rows

Duplicate row counts across the full dataset for both files.

In [17]:
print('--- Duplicate Rows ---\n')
print(f'{"Metric":<25} {"October":>15} {"November":>15}')
print(f'{"-" * 55}')
print(f'{"Duplicate rows":<25} {oct_stats["duplicates"]:>15,} {nov_stats["duplicates"]:>15,}')
print(f'{"Duplicate %":<25} {oct_stats["duplicates"]/oct_stats["total_rows"]*100:>14.3f}% {nov_stats["duplicates"]/nov_stats["total_rows"]*100:>14.3f}%')

--- Duplicate Rows ---

Metric                            October        November
-------------------------------------------------------
Duplicate rows                     30,219         100,510
Duplicate %                        0.071%          0.149%


### Duplicate Rows Observations

| Metric | October | November |
|---|---|---|
| Duplicate rows | 30,219 | 100,510 |
| Duplicate % | 0.071% | 0.149% |

* Duplicates exist in both files but are a very small percentage of the total rows
* November has more duplicates than October, both in count and slightly in percentage
* A duplicate means the exact same user performed the exact same action on the exact same product at the exact same time, which should not be possible
* These rows will be removed during preprocessing
* Our earlier estimate from the 100,000 row sample (17 duplicates) was much lower than the real number. This might be because duplicates are rare enough that a small sample underestimates them

## Unique Counts

Unique counts for users, products, brands, and categories across the full dataset.

In [18]:
print('--- Unique Counts ---\n')
print(f'{"Metric":<25} {"October":>15} {"November":>15}')
print(f'{"-" * 55}')
print(f'{"Unique users":<25} {len(oct_stats["users"]):>15,} {len(nov_stats["users"]):>15,}')
print(f'{"Unique products":<25} {len(oct_stats["products"]):>15,} {len(nov_stats["products"]):>15,}')
print(f'{"Unique brands":<25} {len(oct_stats["brands"]):>15,} {len(nov_stats["brands"]):>15,}')
print(f'{"Unique categories":<25} {len(oct_stats["categories"]):>15,} {len(nov_stats["categories"]):>15,}')

--- Unique Counts ---

Metric                            October        November
-------------------------------------------------------
Unique users                    3,022,290       3,696,117
Unique products                   166,794         190,662
Unique brands                       3,444           4,200
Unique categories                      13              13


### Unique Counts Observations

| Metric | October | November |
|---|---|---|
| Unique users | 3,022,290 | 3,696,117 |
| Unique products | 166,794 | 190,662 |
| Unique brands | 3,444 | 4,200 |
| Unique categories | 13 | 13 |

* November has more unique users, products, and brands than October, which is consistent with the larger overall size of the November file
* The number of unique categories is identical in both months (13), suggesting the store's category structure is set and consistent
* There are only 13 top-level categories across millions of products. This when we analyze behavior by category, every category should have enough data to produce reliable statistics. (but we will have to look into how many products are in each category)
* The large number of unique products (166k-190k) relative to unique brands (3-4k) may suggest most brands have many products in the store (but we have to take into account the large number of rows that dont have a listed brand)
* The large number of unique users (3-3.7 million) is important for the collaborative filtering component. More users means more behavioral patterns to learn from

## Time Range

Date range and unique days across the full dataset for both files.

In [19]:
# Format timestamps for display by removing the UTC offset
oct_min = str(oct_stats["min_time"])[:19]
oct_max = str(oct_stats["max_time"])[:19]
nov_min = str(nov_stats["min_time"])[:19]
nov_max = str(nov_stats["max_time"])[:19]

oct_avg_per_day = oct_stats['total_rows'] / len(oct_stats['unique_days'])
nov_avg_per_day = nov_stats['total_rows'] / len(nov_stats['unique_days'])

print('--- Time Range ---\n')
print(f'{"Metric":<20} {"October":>22} {"November":>22}')
print(f'{"-" * 64}')
print(f'{"Earliest event":<20} {oct_min:>22} {nov_min:>22}')
print(f'{"Latest event":<20} {oct_max:>22} {nov_max:>22}')
print(f'{"Unique days":<20} {len(oct_stats["unique_days"]):>22,} {len(nov_stats["unique_days"]):>22,}')
print(f'{"Avg rows per day":<20} {oct_stats["total_rows"]/len(oct_stats["unique_days"]):>22,.0f} {nov_stats["total_rows"]/len(nov_stats["unique_days"]):>22,.0f}')

print(f'\n--- Daily Activity Comparison ---\n')
print(f'November has {nov_avg_per_day/oct_avg_per_day*100 - 100:.1f}% more average daily activity than October')

--- Time Range ---

Metric                              October               November
----------------------------------------------------------------
Earliest event          2019-10-01 00:00:00    2019-11-01 00:00:00
Latest event            2019-10-31 23:59:59    2019-11-30 23:59:59
Unique days                              31                     30
Avg rows per day                  1,369,315              2,250,066

--- Daily Activity Comparison ---

November has 64.3% more average daily activity than October


### Time Range Observations

| Metric | October | November |
|---|---|---|
| Earliest event | 2019-10-01 00:00:00 | 2019-11-01 00:00:00 |
| Latest event | 2019-10-31 23:59:59 | 2019-11-30 23:59:59 |
| Unique days | 31 | 30 |
| Avg rows per day | 1,369,315 | 2,250,066 |

* Both files cover exactly one full calendar month with no gaps in coverage
* November has a higher average number of events per day (2,250,066 vs 1,369,315), roughly 64% more daily activity than October
* Whether this higher activity is consistent across all days or concentrated around specific dates like Black Friday (November 29th) will be explored in the user behavior analysis notebook

## Purchase Behavior

Users and products with at least one purchase across the full dataset.

In [20]:
print('--- Purchase Behavior ---\n')
print(f'{"Metric":<35} {"October":>15} {"November":>15}')
print(f'{"-" * 65}')
print(f'{"Users with at least one purchase":<35} {len(oct_stats["users_with_purchase"]):>15,} {len(nov_stats["users_with_purchase"]):>15,}')
print(f'{"Total unique users":<35} {len(oct_stats["users"]):>15,} {len(nov_stats["users"]):>15,}')
print(f'{"% users who purchased":<35} {len(oct_stats["users_with_purchase"])/len(oct_stats["users"])*100:>14.1f}% {len(nov_stats["users_with_purchase"])/len(nov_stats["users"])*100:>14.1f}%')
print(f'\n{"Products with at least one purchase":<35} {len(oct_stats["products_with_purchase"]):>15,} {len(nov_stats["products_with_purchase"]):>15,}')
print(f'{"Total unique products":<35} {len(oct_stats["products"]):>15,} {len(nov_stats["products"]):>15,}')
print(f'{"% products that were purchased":<35} {len(oct_stats["products_with_purchase"])/len(oct_stats["products"])*100:>14.1f}% {len(nov_stats["products_with_purchase"])/len(nov_stats["products"])*100:>14.1f}%')

--- Purchase Behavior ---

Metric                                      October        November
-----------------------------------------------------------------
Users with at least one purchase            347,118         441,638
Total unique users                        3,022,290       3,696,117
% users who purchased                         11.5%           11.9%

Products with at least one purchase          42,241          52,054
Total unique products                       166,794         190,662
% products that were purchased                25.3%           27.3%


### Purchase Behavior Observations

| Metric | October | November |
|---|---|---|
| Users with at least one purchase | 347,118 | 441,638 |
| Total unique users | 3,022,290 | 3,696,117 |
| % users who purchased | 11.5% | 11.9% |
| Products with at least one purchase | 42,241 | 52,054 |
| Total unique products | 166,794 | 190,662 |
| % products that were purchased | 25.3% | 27.3% |

* Only 11.5% of October users and 11.9% of November users made at least one purchase in either month, the majority of users are browsers only
* The purchase rate is consistent across both months (11.5% vs 11.9%), maybe suggesting stable buying behavior despite the difference in overall activity
* 25.3% of October products and 27.3% of November products were purchased at least once. Meaning roughly 75% of products were only viewed or added to cart but never bought
* These numbers are important for the collaborative filtering component. With only 11.5% of users having purchase history, the recommendation system will need to handle sparse data carefully
* The consistency between months in both metrics suggests these are stable characteristics of this platform's user base

## Price Statistics

Price distribution and zero price counts across the full dataset.

In [21]:
oct_price_arr = np.array(oct_stats['price_sample'])
nov_price_arr = np.array(nov_stats['price_sample'])

# Format price values with dollar sign included inside the width
def fmt_price(value):
    return f'${value:,.2f}'

print('--- Price Statistics ---\n')
print(f'{"Metric":<25} {"October":>15} {"November":>15}')
print(f'{"-" * 55}')
print(f'{"Min price":<25} {fmt_price(oct_stats["price_min"]):>15} {fmt_price(nov_stats["price_min"]):>15}')
print(f'{"Max price":<25} {fmt_price(oct_stats["price_max"]):>15} {fmt_price(nov_stats["price_max"]):>15}')
print(f'{"Mean price":<25} {fmt_price(oct_stats["price_sum"]/oct_stats["price_count"]):>15} {fmt_price(nov_stats["price_sum"]/nov_stats["price_count"]):>15}')
print(f'{"Median price":<25} {fmt_price(np.median(oct_price_arr)):>15} {fmt_price(np.median(nov_price_arr)):>15}')
print(f'{"Std deviation":<25} {fmt_price(np.std(oct_price_arr)):>15} {fmt_price(np.std(nov_price_arr)):>15}')
print(f'\n{"Zero price rows":<25} {oct_stats["zero_prices"]:>15,} {nov_stats["zero_prices"]:>15,}')
print(f'{"Zero price %":<25} {oct_stats["zero_prices"]/oct_stats["total_rows"]*100:>14.3f}% {nov_stats["zero_prices"]/nov_stats["total_rows"]*100:>14.3f}%')
print(f'{"Zero price products":<25} {len(oct_stats["zero_price_products"]):>15,} {len(nov_stats["zero_price_products"]):>15,}')

--- Price Statistics ---

Metric                            October        November
-------------------------------------------------------
Min price                           $0.00           $0.00
Max price                       $2,574.07       $2,574.07
Mean price                        $290.32         $292.46
Median price                      $163.18         $169.35
Std deviation                     $357.91         $355.55

Zero price rows                    68,673         188,088
Zero price %                       0.162%          0.279%
Zero price products                11,095          13,235


### Price Statistics Observations

| Metric | October | November |
|---|---|---|
| Min price | \$0.00 | \$0.00 |
| Max price | \$2,574.07 | \$2,574.07 |
| Mean price | \$290.32 | \$292.46 |
| Median price | \$163.18 | \$169.35 |
| Std deviation | \$357.91 | \$355.55 |
| Zero price rows | 68,673 | 188,088 |
| Zero price % | 0.162% | 0.279% |
| Zero price products | 11,095 | 13,235 |

* Both files have the exact same max price (\$2,574.07), which may indicate a price cap in the source system
* The mean price (290-292) is significantly higher than the median price (163-169), indicating the price distribution is right-skewed, a small number of expensive products pull the mean up
* Price statistics are very consistent between months, suggesting stable pricing across the platform
* Zero prices exist in both files but are a small percentage of total rows (0.162% and 0.279%)
* Zero prices affect a notable number of distinct products (11,095 and 13,235), these will have to be handled.
* November has almost 3x more zero price rows than October (188,088 vs 68,673) despite being only 1.6x larger. This may be worth investigating further in the user behavior analysis notebook.

## Cross-File Comparison

Comparing users and sessions that appear in both October and November to 
understand overlap between the two months.

In [22]:
# Find users and sessions appearing in both files
users_in_both    = oct_stats['users'] & nov_stats['users']
products_in_both = oct_stats['products'] & nov_stats['products']
brands_in_both   = oct_stats['brands'] & nov_stats['brands']

print('--- Cross-File Comparison ---\n')
print(f'{"Metric":<35} {"Count":>15} {"% of October":>15} {"% of November":>15}')
print(f'{"-" * 80}')
print(f'{"Users in both months":<35} {len(users_in_both):>15,} {len(users_in_both)/len(oct_stats["users"])*100:>14.1f}% {len(users_in_both)/len(nov_stats["users"])*100:>14.1f}%')
print(f'{"Products in both months":<35} {len(products_in_both):>15,} {len(products_in_both)/len(oct_stats["products"])*100:>14.1f}% {len(products_in_both)/len(nov_stats["products"])*100:>14.1f}%')
print(f'{"Brands in both months":<35} {len(brands_in_both):>15,} {len(brands_in_both)/len(oct_stats["brands"])*100:>14.1f}% {len(brands_in_both)/len(nov_stats["brands"])*100:>14.1f}%')

--- Cross-File Comparison ---

Metric                                        Count    % of October   % of November
--------------------------------------------------------------------------------
Users in both months                      1,401,758           46.4%           37.9%
Products in both months                     150,580           90.3%           79.0%
Brands in both months                         3,342           97.0%           79.6%


### Cross-File Comparison Observations

| Metric | Count | % of October | % of November |
|---|---|---|---|
| Users in both months | 1,401,758 | 46.4% | 37.9% |
| Products in both months | 150,580 | 90.3% | 79.0% |
| Brands in both months | 3,342 | 97.0% | 79.6% |

* About 46% of October users returned in November, meaning more than half of October users did not come back the following month
* November has a larger unique user base. Only 37.9% of November users were also active in October, meaning a large portion of November users are new or returning after a longer absence
* Products and brands have much higher overlap between months (90% and 97%). So the store's catalog is largely stable across both months
* The relatively low user overlap (46%) is important for the collaborative filtering component because it means the model needs to handle users with limited cross-month history
* The high product and brand overlap confirms that combining both months into a single processed file is appropriate. We are not mixing two different catalogs, just two months of activity on the same platform

## Summary of Findings

A complete summary of all data quality issues found across both files. These 
findings directly inform the preprocessing pipeline in `02_preprocessing.ipynb`.

In [23]:
print('--- Full Dataset Summary ---\n')
print(f'{"Metric":<35} {"October":>15} {"November":>15}')
print(f'{"-" * 65}')
print(f'{"Total rows":<35} {oct_stats["total_rows"]:>15,} {nov_stats["total_rows"]:>15,}')
print(f'{"Total sessions":<35} {len(oct_stats["sessions"]):>15,} {len(nov_stats["sessions"]):>15,}')
print(f'{"Total unique users":<35} {len(oct_stats["users"]):>15,} {len(nov_stats["users"]):>15,}')
print(f'{"Total unique products":<35} {len(oct_stats["products"]):>15,} {len(nov_stats["products"]):>15,}')
print(f'{"Total unique brands":<35} {len(oct_stats["brands"]):>15,} {len(nov_stats["brands"]):>15,}')
print(f'{"Total unique categories":<35} {len(oct_stats["categories"]):>15,} {len(nov_stats["categories"]):>15,}')

print(f'\n--- Data Quality Issues ---\n')
print(f'{"Missing category_code":<35} {oct_stats["missing"].get("category_code", 0):>15,} {nov_stats["missing"].get("category_code", 0):>15,}')
print(f'{"Missing brand":<35} {oct_stats["missing"].get("brand", 0):>15,} {nov_stats["missing"].get("brand", 0):>15,}')
print(f'{"Null session IDs":<35} {oct_stats["null_sessions"]:>15,} {nov_stats["null_sessions"]:>15,}')
print(f'{"Duplicate rows":<35} {oct_stats["duplicates"]:>15,} {nov_stats["duplicates"]:>15,}')
print(f'{"Zero price rows":<35} {oct_stats["zero_prices"]:>15,} {nov_stats["zero_prices"]:>15,}')

# Combined totals
total_rows     = oct_stats['total_rows'] + nov_stats['total_rows']
total_sessions = len(oct_stats['sessions']) + len(nov_stats['sessions'])
size_ratio     = nov_stats['total_rows'] / oct_stats['total_rows']

print(f'\n--- Combined Totals ---\n')
print(f'{"Total rows (combined)":<35} {total_rows:>15,}')
print(f'{"Total sessions (combined)":<35} {total_sessions:>15,}')
print(f'{"November vs October size ratio":<35} {size_ratio:>15.1f}x')

--- Full Dataset Summary ---

Metric                                      October        November
-----------------------------------------------------------------
Total rows                               42,448,764      67,501,979
Total sessions                            9,244,421      13,776,050
Total unique users                        3,022,290       3,696,117
Total unique products                       166,794         190,662
Total unique brands                           3,444           4,200
Total unique categories                          13              13

--- Data Quality Issues ---

Missing category_code                    13,515,609      21,898,171
Missing brand                             6,117,080       9,224,078
Null session IDs                                  2              10
Duplicate rows                               30,219         100,510
Zero price rows                              68,673         188,088

--- Combined Totals ---

Total rows (combined)           

### Summary Observations

#### Dataset Scale
* The combined dataset contains 109,950,743 rows across 23,020,471 sessions
* There are 13 top-level product categories, 3,444-4,200 brands, and 166,794-190,662 unique products across both months
* November is roughly 1.6x larger than October in terms of rows and sessions

#### Data Quality Issues to Address in Preprocessing

| Issue | Column | Action |
|---|---|---|
| Missing values | `category_code` | Fill with 'unknown' |
| Missing values | `brand` | Fill with 'unknown' |
| Null session IDs | `user_session` | Exclude from session-based sampling |
| Duplicate rows | all columns | Remove exact duplicates |
| Zero prices | `price` | Replace with NaN |
| Text timestamps | `event_time` | Convert to datetime |

#### Key Findings for Later Notebooks
* Only 11.5-11.9% of users made at least one purchase. Collaborative filtering will need to handle sparse purchase data
* Cart to purchase conversion rate differs significantly between months (80.2% in October vs 30.3% in November). Time-based features will be important for the recommendation system
* 46.4% of October users returned in November. There is meaningful cross-month user behavior to learn from
* Price distribution is right-skewed with a median of 163-169 and mean of \$290-292. A small number of expensive products pull the mean up